# Modelado de Área de Incendios con KNN

## ¿Cómo funciona KNN?
- KNN es un algoritmo de aprendizaje supervisado que, para predecir el valor de una instancia, busca los *k* vecinos más cercanos en el espacio de características.
- La cercanía se mide mediante una métrica de distancia (euclídea, Manhattan, etc.).
- En regresión, la predicción es la media (o ponderación) de los valores de los vecinos.

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


## 1. Carga de Datos y Transformación del Area
Cargamos el dataset y se aplicamos **transformación logarítmica** a la variable objetivo (`area`) para reducir asimetrías y valores extremos.


In [18]:
data = pd.read_csv('forestfires.csv')  # Asume forestfires.csv en el directorio
data['area_log'] = np.log1p(data['area'])  # Transformación: log(1 + area)
y = data['area_log']  # El area log-transformada
X = data.drop(['area', 'area_log'], axis=1)  # Resto de atributos
display(X.head())
display(y.head())

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0


0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: area_log, dtype: float64

## 2. Preprocesamiento y Pipeline 
- Usamos **RobustScaler**: Escala variables numéricas reduciendo el efecto de outliers.
- Creamos un **Pipeline** que aplicando el preprocesamiento y luego KNN de regersión.
- OneHot con **get_dummies()**: Codifica las variables categóricas transformandolas a numericas.


In [19]:
# Creacion del pipeline con preprocesamiento RobustScaler() y modelo de regresion KNeighborsRegressor()
model = Pipeline([
    ("scaler", RobustScaler()),
    ("regressor", KNeighborsRegressor())
])
# Codificación one-hot de 'month' y 'day'
X = pd.get_dummies(X, columns=['month', 'day'], drop_first=True).astype(float)
print(X)

       X    Y  FFMC    DMC     DC   ISI  temp    RH  wind  rain  ...  \
0    7.0  5.0  86.2   26.2   94.3   5.1   8.2  51.0   6.7   0.0  ...   
1    7.0  4.0  90.6   35.4  669.1   6.7  18.0  33.0   0.9   0.0  ...   
2    7.0  4.0  90.6   43.7  686.9   6.7  14.6  33.0   1.3   0.0  ...   
3    8.0  6.0  91.7   33.3   77.5   9.0   8.3  97.0   4.0   0.2  ...   
4    8.0  6.0  89.3   51.3  102.2   9.6  11.4  99.0   1.8   0.0  ...   
..   ...  ...   ...    ...    ...   ...   ...   ...   ...   ...  ...   
512  4.0  3.0  81.6   56.7  665.6   1.9  27.8  32.0   2.7   0.0  ...   
513  2.0  4.0  81.6   56.7  665.6   1.9  21.9  71.0   5.8   0.0  ...   
514  7.0  4.0  81.6   56.7  665.6   1.9  21.2  70.0   6.7   0.0  ...   
515  1.0  4.0  94.4  146.0  614.7  11.3  25.6  42.0   4.0   0.0  ...   
516  6.0  3.0  79.5    3.0  106.7   1.1  11.8  31.0   4.5   0.0  ...   

     month_may  month_nov  month_oct  month_sep  day_mon  day_sat  day_sun  \
0          0.0        0.0        0.0        0.0      0.0 

## 3. Division de los datos de entrenamiento y validacion
- Utilizamos `train_test_split()` para separar los datos en un 80% de `entrenamiento` y 20% para realizar los `test`


In [4]:
# División train/test, 80% entrenamineto y 20% para el test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Validación Cruzada y Búsqueda de Hiperparámetros
- Se utiliza **RepeatedKFold** con 10 folds y 30 repeticiones para tener una validación robusta.
- Probamos con distintos valores de `n_neighbors(k)`, `weights` y `metric`.
- **GridSearchCV** con métrica `neg_mean_squared_error` porque asume que el MSE mas alto es el
     mejor, asi que usamos el negativo.


In [5]:
# Validacion cruzada
cv = RepeatedKFold(n_splits=5, n_repeats=30, random_state=42)
model, cv

# Seleccion de hiperparámetros
param_grid = {
    'regressor__n_neighbors': range(2, 25),
    'regressor__weights': ['uniform', 'distance'],
    'regressor__metric': ['euclidean', 'manhattan']
}

# Búsqueda de hiperparámetros
grid_search = GridSearchCV(model, param_grid, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)
print('Mejores parámetros:', grid_search.best_params_)

Mejores parámetros: {'regressor__metric': 'euclidean', 'regressor__n_neighbors': 24, 'regressor__weights': 'distance'}


## 5. Evaluación Final
- Se evaluúa el mejor modelo sobre todo el dataset (predicciones y anti-transformación).
- Cálculo de **MSE**, **MAE** y **R²** en la escala original:
- **MSE** (Mean Squared Error): promedio de los cuadrados de los errores (hectareas²).
- **MAE** (Mean Absolute Error): promedio de los valores absolutos de los errores en hectareas.
- **R²**  (coeficiente de determinación): proporción de la varianza explicada por el modelo (adimensional, de 0 a 1).
- **Error absoluto** diferencia |predicción – valor real|, medido en hectáreas.
- **% de error** (error absoluto / valor real) × 100, un porcentaje que indica el tamaño del error relativo al valor real.


In [12]:
# Elegimos el mejor modelo
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_pred_result = np.expm1(y_pred)
y_true_result = np.expm1(y_test)

# Calculamos las metricas
mse = mean_squared_error(y_true_result, y_pred_result)
mae = mean_absolute_error(y_true_result, y_pred_result)
r2 = r2_score(y_true_result, y_pred_result)

print(f"\n--- Métricas sobre test ---")
print(f'MSE: {mse:.2f}')
print(f'MAE: {mae:.2f}')
print(f'R²: {r2:.2f}')


--- Métricas sobre test ---
MSE: 12090.00
MAE: 19.76
R²: -0.03


## 6. Predicciones de Ejemplo
Cogemos 5 predicciones aleatorias del dataset y cálculamos la prediccion del aera quemada, el error absoluto y el error relativo de cada una.


In [ ]:
# Predicciones de ejemplo
print("\n=== Predicciones de Ejemplo ===")
random_indices = random.sample(range(len(X_test)), 5)

for i, idx in enumerate(random_indices):
    sample = X_test.iloc[idx:idx+1]
    real_value = y_true_result.iloc[idx]
    prediction = np.expm1(best_model.predict(sample))[0]
    error = abs(prediction - real_value)
    
    print(f"\nMuestra {i+1} (índice {idx}):")
    print(f"- Real: {real_value:.2f} hectáreas")
    print(f"- Predicción: {prediction:.2f} hectáreas")
    print(f"- Error absoluto: {error:.2f} hectáreas")
    print(f"- % Error: {(error/(real_value+1e-6))*100:.1f}%")


=== Predicciones de Ejemplo ===

Muestra 1 (índice 81):
- Real: 95.18 hectáreas
- Predicción: 4.68 hectáreas
- Error absoluto: 90.50 hectáreas
- % Error: 95.1%

Muestra 2 (índice 89):
- Real: 0.52 hectáreas
- Predicción: 2.30 hectáreas
- Error absoluto: 1.78 hectáreas
- % Error: 343.1%

Muestra 3 (índice 10):
- Real: 0.00 hectáreas
- Predicción: 1.97 hectáreas
- Error absoluto: 1.97 hectáreas
- % Error: 196721789.0%

Muestra 4 (índice 92):
- Real: 13.70 hectáreas
- Predicción: 1.86 hectáreas
- Error absoluto: 11.84 hectáreas
- % Error: 86.4%

Muestra 5 (índice 97):
- Real: 1.52 hectáreas
- Predicción: 1.13 hectáreas
- Error absoluto: 0.39 hectáreas
- % Error: 25.7%
